In [ ]:
import sqlite3
from project_starter import init_database
from sqlalchemy import create_engine

# Database initialization is a separate setup step, never part of Run All.
# Uncomment init_database to reset the database to its starting state.
conn = sqlite3.connect("munder_difflin.db")
db_engine = create_engine("sqlite:///munder_difflin.db")
# init_database(db_engine)

In [7]:
import pandas as pd

# Query the sqlite_master system table
query = "SELECT name FROM sqlite_master WHERE type='table';"
df_tables = pd.read_sql_query(query, conn)
df_tables

,name
0,transactions
1,quote_requests
2,quotes
3,inventory
4,paper_supplies


In [8]:
pd.read_sql("select * from transactions", conn)

,id,item_name,transaction_type,units,price,transaction_date
0,None,NaN,sales,NaN,50000.00,2025-01-01T00:00:00
1,None,Paper plates,stock_orders,748.0,74.80,2025-01-01T00:00:00
2,None,100 lb cover stock,stock_orders,636.0,318.00,2025-01-01T00:00:00
3,None,Glossy paper,stock_orders,587.0,117.40,2025-01-01T00:00:00
4,None,Rolls of banner paper (36-inch width),stock_orders,546.0,1365.00,2025-01-01T00:00:00
5,None,Photo paper,stock_orders,423.0,105.75,2025-01-01T00:00:00
6,None,Cardstock,stock_orders,595.0,89.25,2025-01-01T00:00:00
7,None,Colored paper,stock_orders,788.0,78.80,2025-01-01T00:00:00
8,None,80 lb text paper,stock_orders,249.0,99.60,2025-01-01T00:00:00
9,None,Large poster paper (24x36 inches),stock_orders,699.0,699.00,2025-01-01T00:00:00


In [ ]:
pd.read_sql("select * from quote_requests", conn)

In [ ]:
pd.read_sql("select * from quotes", conn)

In [ ]:
pd.read_sql("select * from inventory", conn)

### Configuration

In [1]:
from model_config import create_model

model = create_model()

/Users/jameschang/Downloads/agentic-ai-c4-exercises-demos/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### RequestUnderstandingAgent

In [2]:
from request_schema import ParsedItems
from request_schema import paper_supplies
from request_understanding_agent import RequestUnderstandingAgent

request_understanding_agent = RequestUnderstandingAgent(model=model)
result = request_understanding_agent.run(
    """I would like to request the following paper supplies for the ceremony: 

- 200 sheets of A4 glossy paper
- 100 sheets of heavy cardstock (white)
- 100 sheets of colored paper (assorted colors)

I need these supplies delivered by April 15, 2025. Thank you."""
)

parsed = (
    ParsedItems.model_validate_json(result)
    if isinstance(result, str)
    else ParsedItems.model_validate(result)
)

if any(item.current_stock is not None for item in parsed.items):
    raise ValueError(
        "The parsing agent must return current_stock as null; "
        "inventory tools populate it later."
    )

print(parsed.model_dump_json(indent=2))

╭───────────────────────────────────── New run - request_understanding_agent ─────────────────────────────────────╮
│                                                                                                                 │
│ I would like to request the following paper supplies for the ceremony:                                          │
│                                                                                                                 │
│ - 200 sheets of A4 glossy paper                                                                                 │
│ - 100 sheets of heavy cardstock (white)                                                                         │
│ - 100 sheets of colored paper (assorted colors)                                                                 │
│                                                                                                                 │
│ I need these supplies delivered by April 15, 2025. Thank you.                                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.6-luna ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer':                                                         │
│ '{"message_scope":"in_scope","mood":null,"items":[{"customer_description":"200 sheets of A4 glossy              │
│ paper","match_status":"matched","item_name":"Glossy                                                             │
│ paper","category":"paper","unit_price":0.2,"min_stock_level":147,"units":200,"quantity_unit":"sheet","package_s │
│ ize":null,"package_unit":null,"current_stock":null,"clarification_question":null},{"customer_description":"100  │
│ sheets of heavy cardstock                                                                                       │
│ (white)","match_status":"matched","item_name":"Cardstock","category":"paper","unit_price":0.15,"min_stock_level │
│ ":148,"units":100,"quantity_unit":"sheet","package_size":null,"package_unit":null,"current_stock":null,"clarifi │
│ cation_question":null},{"customer_description":"100 sheets of colored paper (assorted                           │
│ colors)","match_status":"matched","item_name":"Colored                                                          │
│ paper","category":"paper","unit_price":0.1,"min_stock_level":143,"units":100,"quantity_unit":"sheet","package_s │
│ ize":null,"package_unit":null,"current_stock":null,"clarification_question":null}],"delivery_by":"2025-04-15"," │
│ delivery_date_question":null}'}                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"message_scope":"in_scope","mood":null,"items":|{"customer_description":"200 sheets of A4 glossy 
paper","match_status":"matched","item_name":"Glossy 
paper","category":"paper","unit_price":0.2,"min_stock_level":147,"units":200,"quantity_unit":"sheet","package_size"
:null,"package_unit":null,"current_stock":null,"clarification_question":null},{"customer_description":"100 sheets 
of heavy cardstock 
(white)","match_status":"matched","item_name":"Cardstock","category":"paper","unit_price":0.15,"min_stock_level":14
8,"units":100,"quantity_unit":"sheet","package_size":null,"package_unit":null,"current_stock":null,"clarification_q
uestion":null},{"customer_description":"100 sheets of colored paper (assorted 
colors)","match_status":"matched","item_name":"Colored 
paper","category":"paper","unit_price":0.1,"min_stock_level":143,"units":100,"quantity_unit":"sheet","package_size"
:null,"package_unit":null,"current_stock":null,"clarification_question":null}],"delivery_by":"2025-04-15","delivery
_date_question":null}

Final answer: {"message_scope":"in_scope","mood":null,"items":[{"customer_description":"200 sheets of A4 glossy 
paper","match_status":"matched","item_name":"Glossy 
paper","category":"paper","unit_price":0.2,"min_stock_level":147,"units":200,"quantity_unit":"sheet","package_size"
:null,"package_unit":null,"current_stock":null,"clarification_question":null},{"customer_description":"100 sheets 
of heavy cardstock 
(white)","match_status":"matched","item_name":"Cardstock","category":"paper","unit_price":0.15,"min_stock_level":14
8,"units":100,"quantity_unit":"sheet","package_size":null,"package_unit":null,"current_stock":null,"clarification_q
uestion":null},{"customer_description":"100 sheets of colored paper (assorted 
colors)","match_status":"matched","item_name":"Colored 
paper","category":"paper","unit_price":0.1,"min_stock_level":143,"units":100,"quantity_unit":"sheet","package_size"
:null,"package_unit":null,"current_stock":null,"clarification_question":null}],"delivery_by":"2025-04-15","delivery
_date_question":null}

[Step 1: Duration 4.29 seconds| Input tokens: 6,472 | Output tokens: 265]

{
  "message_scope": "in_scope",
  "mood": null,
  "items": [
    {
      "customer_description": "200 sheets of A4 glossy paper",
      "match_status": "matched",
      "item_name": "Glossy paper",
      "category": "paper",
      "unit_price": 0.2,
      "min_stock_level": 147,
      "units": 200,
      "quantity_unit": "sheet",
      "package_size": null,
      "package_unit": null,
      "current_stock": null,
      "clarification_question": null
    },
    {
      "customer_description": "100 sheets of heavy cardstock (white)",
      "match_status": "matched",
      "item_name": "Cardstock",
      "category": "paper",
      "unit_price": 0.15,
      "min_stock_level": 148,
      "units": 100,
      "quantity_unit": "sheet",
      "package_size": null,
      "package_unit": null,
      "current_stock": null,
      "clarification_question": null
    },
    {
      "customer_description": "100 sheets of colored paper (assorted colors)",
      "match_status": "matched",
      "item_na

#### InventoryManagementAgent

In [3]:
from inventory_management_agent import InventoryManagementAgent

inventory_management_agent = InventoryManagementAgent(model=model)
request_date = "04/03/25"
task = (
    "Plan stock and replenishment for this request.\n"
    f"Request date (MM/DD/YY): {request_date}\n"
    "Structured request:\n" + parsed.model_dump_json()
)
result = inventory_management_agent.run(task)

╭───────────────────────────────────── New run - inventory_management_agent ──────────────────────────────────────╮
│                                                                                                                 │
│ Plan stock and replenishment for this request.                                                                  │
│ Request date (MM/DD/YY): 04/03/25                                                                               │
│ Structured request:                                                                                             │
│ {"message_scope":"in_scope","mood":null,"items":[{"customer_description":"200 sheets of A4 glossy               │
│ paper","match_status":"matched","item_name":"Glossy                                                             │
│ paper","category":"paper","unit_price":0.2,"min_stock_level":147,"units":200,"quantity_unit":"sheet","package_s │
│ ize":null,"package_unit":null,"current_stock":null,"clarification_question":null},{"customer_description":"100  │
│ sheets of heavy cardstock                                                                                       │
│ (white)","match_status":"matched","item_name":"Cardstock","category":"paper","unit_price":0.15,"min_stock_level │
│ ":148,"units":100,"quantity_unit":"sheet","package_size":null,"package_unit":null,"current_stock":null,"clarifi │
│ cation_question":null},{"customer_description":"100 sheets of colored paper (assorted                           │
│ colors)","match_status":"matched","item_name":"Colored                                                          │
│ paper","category":"paper","unit_price":0.1,"min_stock_level":143,"units":100,"quantity_unit":"sheet","package_s │
│ ize":null,"package_unit":null,"current_stock":null,"clarification_question":null}],"delivery_by":"2025-04-15"," │
│ delivery_date_question":null}                                                                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.6-luna ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'plan_replenishment' with arguments: {'parsed': {'message_scope': 'in_scope', 'mood': None,       │
│ 'items': [{'customer_description': '200 sheets of A4 glossy paper', 'match_status': 'matched', 'item_name':     │
│ 'Glossy paper', 'category': 'paper', 'unit_price': 0.2, 'min_stock_level': 147, 'units': 200, 'quantity_unit':  │
│ 'sheet', 'package_size': None, 'package_unit': None, 'current_stock': None, 'clarification_question': None},    │
│ {'customer_description': '100 sheets of heavy cardstock (white)', 'match_status': 'matched', 'item_name':       │
│ 'Cardstock', 'category': 'paper', 'unit_price': 0.15, 'min_stock_level': 148, 'units': 100, 'quantity_unit':    │
│ 'sheet', 'package_size': None, 'package_unit': None, 'current_stock': None, 'clarification_question': None},    │
│ {'customer_description': '100 sheets of colored paper (assorted colors)', 'match_status': 'matched',            │
│ 'item_name': 'Colored paper', 'category': 'paper', 'unit_price': 0.1, 'min_stock_level': 143, 'units': 100,     │
│ 'quantity_unit': 'sheet', 'package_size': None, 'package_unit': None, 'current_stock': None,                    │
│ 'clarification_question': None}], 'delivery_by': '2025-04-15', 'delivery_date_question': None}, 'as_of_date':   │
│ '04/03/25'}                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |{'item_name': 'Glossy paper', 'unit_price': 0.2, 'requested_units': 200, 'current_stock': 587, 
'ordered_units': 0, 'ship_date': '2025-04-03', 'on_time': True, 'can_fulfill': True, 'reject_reason': None}, 
{'item_name': 'Cardstock', 'unit_price': 0.15, 'requested_units': 100, 'current_stock': 595, 'ordered_units': 0, 
'ship_date': '2025-04-03', 'on_time': True, 'can_fulfill': True, 'reject_reason': None}, {'item_name': 'Colored 
paper', 'unit_price': 0.1, 'requested_units': 100, 'current_stock': 788, 'ordered_units': 0, 'ship_date': 
'2025-04-03', 'on_time': True, 'can_fulfill': True, 'reject_reason': None}]

[Step 1: Duration 3.56 seconds| Input tokens: 1,543 | Output tokens: 270]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '[{"item_name":"Glossy                                  │
│ paper","unit_price":0.2,"requested_units":200,"current_stock":587,"ordered_units":0,"ship_date":"2025-04-03","o │
│ n_time":true,"can_fulfill":true,"reject_reason":null},{"item_name":"Cardstock","unit_price":0.15,"requested_uni │
│ ts":100,"current_stock":595,"ordered_units":0,"ship_date":"2025-04-03","on_time":true,"can_fulfill":true,"rejec │
│ t_reason":null},{"item_name":"Colored                                                                           │
│ paper","unit_price":0.1,"requested_units":100,"current_stock":788,"ordered_units":0,"ship_date":"2025-04-03","o │
│ n_time":true,"can_fulfill":true,"reject_reason":null}]'}                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |{"item_name":"Glossy 
paper","unit_price":0.2,"requested_units":200,"current_stock":587,"ordered_units":0,"ship_date":"2025-04-03","on_ti
me":true,"can_fulfill":true,"reject_reason":null},{"item_name":"Cardstock","unit_price":0.15,"requested_units":100,
"current_stock":595,"ordered_units":0,"ship_date":"2025-04-03","on_time":true,"can_fulfill":true,"reject_reason":nu
ll},{"item_name":"Colored 
paper","unit_price":0.1,"requested_units":100,"current_stock":788,"ordered_units":0,"ship_date":"2025-04-03","on_ti
me":true,"can_fulfill":true,"reject_reason":null}]

Final answer: [{"item_name":"Glossy 
paper","unit_price":0.2,"requested_units":200,"current_stock":587,"ordered_units":0,"ship_date":"2025-04-03","on_ti
me":true,"can_fulfill":true,"reject_reason":null},{"item_name":"Cardstock","unit_price":0.15,"requested_units":100,
"current_stock":595,"ordered_units":0,"ship_date":"2025-04-03","on_time":true,"can_fulfill":true,"reject_reason":nu
ll},{"item_name":"Colored 
paper","unit_price":0.1,"requested_units":100,"current_stock":788,"ordered_units":0,"ship_date":"2025-04-03","on_ti
me":true,"can_fulfill":true,"reject_reason":null}]

[Step 2: Duration 2.54 seconds| Input tokens: 3,681 | Output tokens: 456]

#### QuotingAgent

In [5]:
from pricing import price_quote

quote = price_quote(parsed)
quote

Quote(lines=[QuoteLine(item_name='Glossy paper', quantity=200, unit='sheet', unit_price=0.25, list_total=50.0, sale_price=50.0), QuoteLine(item_name='Cardstock', quantity=100, unit='sheet', unit_price=0.1875, list_total=18.75, sale_price=18.75), QuoteLine(item_name='Colored paper', quantity=100, unit='sheet', unit_price=0.125, list_total=12.5, sale_price=12.5)], unpriced=[], subtotal=81.25, discount_rate=0.0, discount_amount=0.0, total=81.25, rationale='Prices are our standard list prices per sheet, piece or roll. Orders of $500 or more receive a 5% bulk discount. Total: $81.25.', cost=65.0, gross_margin=0.2)

#### Orchestrator

In [ ]:
from orchestrator import Orchestrator

# Stop at the quote so these demos don't record sales in the database.
orchestrator = Orchestrator(model=model, auto_confirm=False)
request_date = "2025-04-03"

In [ ]:
# 1. Normal case
orchestrator.start_conversation()
result = orchestrator.process_customer_message(
    "I would like to place an order for 500 sheets of high-quality glossy paper and 300 sheets of sturdy cardstock for our upcoming show. We need the supplies delivered by April 15, 2025.",
    request_date,
    request_id=1,
)

print("Status:", result["status"])
print("Assistant:", result["reply"])

In [ ]:
# 2. Missing information, then the customer's answer in the same conversation
orchestrator.start_conversation()
result = orchestrator.process_customer_message(
    "I would like to order A4 paper.", request_date, request_id=2
)
print("Status:", result["status"])
print("Assistant:", result["reply"])

result = orchestrator.process_customer_message(
    "2 reams, delivered by April 15, 2025.", request_date, request_id=2
)
print("Status:", result["status"])
print("Assistant:", result["reply"])

In [ ]:
# 3. Out-of-scope question
orchestrator.start_conversation()
result = orchestrator.process_customer_message(
    "I would like to book a flight to Paris.", request_date, request_id=3
)

print("Status:", result["status"])
print("Assistant:", result["reply"])